In [1]:
# ================================================================
# HALLUVERSE-M3 — GEMMA3:1B DIALOGUE SUMMARIZATION (RESUMABLE)
# ================================================================
#
# Task: Gemma reads a dialogue and writes its OWN summary
# (English dialogue -> English summary, Nepali dialogue -> Nepali summary).
# We save Gemma's summary next to the reference (human) summary so you
# can grade each pair afterward for hallucination (same as the other
# five domains).
#
# Source file is already cleaned to PAIRED rows only
# (Hexagon-Dataset-HalluVerseM3_PAIRED.csv) — no further filtering needed.
# ================================================================

import os
import time
from pathlib import Path

import pandas as pd
import requests


# ================================================================
# 1. SETTINGS
# ================================================================

FOLDER = Path(
    r"C:\Users\LENOVO\OneDrive - London Metropolitan University\Documents\hexagon-research"
)

SOURCE_FILE = FOLDER / "Hexagon-Dataset-HalluVerseM3_PAIRED.csv"

OUTPUT_FILE = FOLDER / "HalluVerseM3_Gemma3_1B_Summaries.csv"

OLLAMA_URL = "http://localhost:11434/api/generate"
MODEL = "gemma3:1b"


# ================================================================
# 2. LOAD SOURCE (already paired — no filtering needed)
# ================================================================

if not SOURCE_FILE.exists():
    raise FileNotFoundError(f"Source file not found:\n{SOURCE_FILE}")

df = pd.read_csv(SOURCE_FILE)

required = [
    "ID", "Dialogue_EN", "Reference_Summary_EN", "Topic_EN",
    "Dialogue_NE", "Reference_Summary_NE", "Topic_NE",
]
missing = [c for c in required if c not in df.columns]
if missing:
    raise ValueError(f"Missing source columns: {missing}")

df["ID_STR"] = df["ID"].astype(str).str.strip()
df = df.drop_duplicates(subset=["ID_STR"]).reset_index(drop=True)

print("Paired rows in source:", len(df))
print("Expected evaluations (paired x 2 languages):", len(df) * 2)


# ================================================================
# 3. LOAD OR CREATE CHECKPOINT
# ================================================================

CHECKPOINT_COLS = [
    "ID", "Language", "Topic",
    "Dialogue", "Reference Summary",
    "Gemma Raw Output", "Gemma Summary",
]

if OUTPUT_FILE.exists():
    results = pd.read_csv(OUTPUT_FILE)
    missing_result = [c for c in CHECKPOINT_COLS if c not in results.columns]
    if missing_result:
        raise ValueError(f"Checkpoint is missing columns: {missing_result}")
    results = results.drop_duplicates(subset=["ID", "Language"], keep="last").reset_index(drop=True)
else:
    results = pd.DataFrame(columns=CHECKPOINT_COLS)
    results.to_csv(OUTPUT_FILE, index=False, encoding="utf-8-sig")

print("Existing saved evaluations:", len(results))
if len(results):
    print(results["Language"].value_counts().to_dict())


# ================================================================
# 4. EXPECTED PAIRS
# ================================================================

expected_pairs = {
    (item_id, language)
    for item_id in df["ID_STR"]
    for language in ["English", "Nepali"]
}


def get_pairs(data):
    return {
        (str(row["ID"]).strip(), str(row["Language"]).strip())
        for _, row in data.iterrows()
    }


# ================================================================
# 5. OLLAMA CHECK
# ================================================================

def wait_for_ollama():
    while True:
        try:
            r = requests.get("http://localhost:11434/api/tags", timeout=10)
            if r.status_code == 200:
                print("Ollama online.")
                return
        except Exception:
            pass
        print("Ollama unavailable. Retrying in 10 seconds...")
        time.sleep(10)


wait_for_ollama()


# ================================================================
# 6. SAFE SAVE
# ================================================================

def safe_save(data):
    temp_file = OUTPUT_FILE.with_suffix(OUTPUT_FILE.suffix + ".tmp")
    while True:
        try:
            data.to_csv(temp_file, index=False, encoding="utf-8-sig")
            os.replace(temp_file, OUTPUT_FILE)
            return
        except PermissionError:
            print("CSV is open/locked. Close it and retrying in 5 seconds...")
            time.sleep(5)


# ================================================================
# 7. GEMMA REQUEST
# ================================================================

def ask_gemma(prompt):
    retry = 0
    wait = 3
    while True:
        try:
            r = requests.post(
                OLLAMA_URL,
                json={
                    "model": MODEL,
                    "prompt": prompt,
                    "stream": False,
                    "keep_alive": "5m",
                    "options": {"temperature": 0},
                },
                timeout=(15, 600),
            )
            if r.status_code == 200:
                return r.json().get("response", "").strip()
            print("Ollama HTTP error:", r.status_code)
        except requests.exceptions.ConnectionError:
            print("Ollama connection lost.")
        except requests.exceptions.Timeout:
            print("Ollama timeout.")
        except Exception as e:
            print("Ollama error:", e)

        retry += 1
        print(f"Retry {retry} in {wait}s...")
        time.sleep(wait)
        wait = min(wait * 2, 30)


# ================================================================
# 8. ONE TARGET
# ================================================================

def evaluate_target(item_id, language, row):

    if language == "English":
        dialogue = str(row["Dialogue_EN"])
        reference_summary = row.get("Reference_Summary_EN", "")
        topic = row.get("Topic_EN", "")

        prompt = f"""You are a dialogue summarization model.

Read the following dialogue and write a concise, ONE-sentence summary.
Only include information stated in the dialogue. Do not add names,
facts, or details that are not present in the dialogue.

Dialogue:
{dialogue}

Summary:"""

    else:
        dialogue = str(row["Dialogue_NE"])
        reference_summary = row.get("Reference_Summary_NE", "")
        topic = row.get("Topic_NE", "")

        prompt = f"""तपाईं संवाद सारांश बनाउने मोडेल हुनुहुन्छ।

तलको संवाद पढेर एक वाक्यमा छोटो सारांश लेख्नुहोस्।
संवादमा नभएका नाम, तथ्य वा विवरणहरू थप नगर्नुहोस् — संवादमा भएको
कुरा मात्र समावेश गर्नुहोस्।

संवाद:
{dialogue}

सारांश:"""

    raw_output = ask_gemma(prompt)
    summary = raw_output.strip()

    return {
        "ID": item_id,
        "Language": language,
        "Topic": topic,
        "Dialogue": dialogue,
        "Reference Summary": reference_summary,
        "Gemma Raw Output": raw_output,
        "Gemma Summary": summary,
    }


# ================================================================
# 9. RESUME LOOP
# ================================================================

while True:

    results = pd.read_csv(OUTPUT_FILE)
    completed_pairs = get_pairs(results)

    missing_pairs = sorted(
        expected_pairs - completed_pairs,
        key=lambda x: (x[0], 0 if x[1] == "English" else 1),
    )

    if not missing_pairs:
        print("\n" + "=" * 70)
        print(f"ALL {len(expected_pairs)} EVALUATIONS COMPLETE")
        print("=" * 70)
        break

    item_id, language = missing_pairs[0]
    row = df[df["ID_STR"] == item_id].iloc[0]

    print(f"\nProcessing: {item_id} | {language}")

    new_row = evaluate_target(item_id, language, row)

    # Recheck before save (protects against double-runs)
    results = pd.read_csv(OUTPUT_FILE)
    current_pairs = get_pairs(results)
    if (item_id, language) in current_pairs:
        print(f"{item_id} | {language} already saved.")
        continue

    results = pd.concat([results, pd.DataFrame([new_row])], ignore_index=True)
    safe_save(results)

    verified = pd.read_csv(OUTPUT_FILE)
    verified_pairs = get_pairs(verified)
    if (item_id, language) not in verified_pairs:
        raise RuntimeError(f"Save verification failed: {item_id} | {language}")

    print(
        f"{item_id} | {language} | "
        f"Saved={len(verified_pairs)} | "
        f"Remaining={len(expected_pairs - verified_pairs)}"
    )


# ================================================================
# 10. FINAL AUDIT
# ================================================================

final_df = pd.read_csv(OUTPUT_FILE)
final_pairs = get_pairs(final_df)
missing_final = expected_pairs - final_pairs
duplicates = int(final_df.duplicated(subset=["ID", "Language"], keep=False).sum())

print("\n" + "=" * 70)
print("FINAL AUDIT")
print("=" * 70)
print("Expected:", len(expected_pairs))
print("Actual:", len(final_pairs))
print("Missing:", len(missing_final))
print("Duplicate rows:", duplicates)
print("\nLanguage counts:")
print(final_df["Language"].value_counts())

if len(missing_final) == 0 and duplicates == 0:
    print("\nCOMPLETE AND VERIFIED")
else:
    print("\nSTILL INCOMPLETE")
    print("First missing:", list(missing_final)[:30])

print("\nFile:")
print(OUTPUT_FILE)

Paired rows in source: 199
Expected evaluations (paired x 2 languages): 398
Existing saved evaluations: 0
Ollama unavailable. Retrying in 10 seconds...
Ollama online.

Processing: train_10022 | English
train_10022 | English | Saved=1 | Remaining=397

Processing: train_10022 | Nepali
train_10022 | Nepali | Saved=2 | Remaining=396

Processing: train_10115 | English
train_10115 | English | Saved=3 | Remaining=395

Processing: train_10115 | Nepali
train_10115 | Nepali | Saved=4 | Remaining=394

Processing: train_10218 | English
train_10218 | English | Saved=5 | Remaining=393

Processing: train_10218 | Nepali
train_10218 | Nepali | Saved=6 | Remaining=392

Processing: train_10287 | English
train_10287 | English | Saved=7 | Remaining=391

Processing: train_10287 | Nepali
train_10287 | Nepali | Saved=8 | Remaining=390

Processing: train_10362 | English
train_10362 | English | Saved=9 | Remaining=389

Processing: train_10362 | Nepali
train_10362 | Nepali | Saved=10 | Remaining=388

Processing: